# Example 3 -- Synthetic Data Generation

An AI agent can generate synthetic data that preserves the *schema* (column
names, dtypes, categorical domains) of a real dataset but contains **no real
values**.

**Use case:** an agent needs realistic-looking data to prototype analysis code,
build visualisations, or validate a pipeline -- without accessing actual
sensitive records.

## 1. Prepare a Sample Customer Dataset

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd

from agent_privacy_layer import PrivacyLayer, UserConfirmation

# Prepare a sample customer dataset.
df = pd.DataFrame(
    {
        "name": ["Alice Smith", "Bob Jones", "Charlie Lee", "Diana Park", "Eve Brown"],
        "email": [
            "alice@example.com",
            "bob@test.org",
            "charlie@mail.com",
            "diana@example.com",
            "eve@test.org",
        ],
        "age": [25, 30, 28, 35, 22],
        "salary": [50_000.0, 60_000.0, 55_000.0, 70_000.0, 45_000.0],
        "department": ["Engineering", "Sales", "Engineering", "Sales", "Engineering"],
        "is_active": [True, True, False, True, False],
    }
)

print(f"Dataset created: {df.shape[0]} rows, {df.shape[1]} columns")

Dataset created: 5 rows, 6 columns


## 2. Random Strategy (Default)

The random strategy is fast and requires no extra dependencies. It generates
synthetic values by sampling from the observed domain of each column.

In [2]:
layer = PrivacyLayer(
    df,
    confirmation=UserConfirmation(dry_run=True),
    synthesis_strategy="random",
)

print("=== Synthetic data (random strategy, same row count) ===")
synthetic_random = layer.synthesize_data()
print(synthetic_random.to_string(index=False))

=== Synthetic data (random strategy, same row count) ===
       name             email  age       salary  department  is_active
  Eve Brown  charlie@mail.com   23 68052.715898       Sales      False
  Bob Jones      bob@test.org   30 61757.734818 Engineering      False
Charlie Lee      eve@test.org   35 62947.953121 Engineering      False
Alice Smith alice@example.com   31 59333.900830       Sales      False
Alice Smith      bob@test.org   35 49208.153599 Engineering      False


In [3]:
# Generate a different number of rows
print("=== Synthetic data (random strategy, 10 rows) ===")
synthetic_random_10 = layer.synthesize_data(n_rows=10)
print(synthetic_random_10.to_string(index=False))

=== Synthetic data (random strategy, 10 rows) ===
       name             email  age       salary  department  is_active
Charlie Lee      eve@test.org   35 60300.991244       Sales      False
 Diana Park diana@example.com   23 63948.584817       Sales      False
Alice Smith diana@example.com   31 69708.376670 Engineering       True
  Eve Brown diana@example.com   23 45537.667550 Engineering       True
  Eve Brown      eve@test.org   30 68993.991234 Engineering      False
 Diana Park alice@example.com   32 65905.161403 Engineering      False
 Diana Park      bob@test.org   34 48614.859790       Sales       True
  Bob Jones alice@example.com   30 59241.826700       Sales      False
 Diana Park  charlie@mail.com   34 59621.138341 Engineering       True
  Bob Jones diana@example.com   24 52431.189998       Sales      False


## 3. Faker Strategy

The Faker strategy produces contextually realistic fake values based on column
name heuristics (e.g., a column named `email` gets plausible fake email addresses).

In [4]:
layer_faker = PrivacyLayer(
    df,
    confirmation=UserConfirmation(dry_run=True),
    synthesis_strategy="faker",
)

print("=== Synthetic data (faker strategy, 5 rows) ===")
synthetic_faker = layer_faker.synthesize_data(n_rows=5)
print(synthetic_faker.to_string(index=False))

=== Synthetic data (faker strategy, 5 rows) ===
            name                   email  age       salary  department  is_active
   Marvin Martin   cooketara@example.org   30 48520.433475 Engineering       True
  Kenneth Turner thomasadams@example.net   30 69961.058743       Sales      False
   Martha Taylor     vjacobs@example.net   34 56743.031970 Engineering      False
    Thomas Evans  wrightdawn@example.net   22 58318.695365       Sales       True
Angelica Andrews     pfields@example.com   29 67835.242229 Engineering       True


## 4. Override Strategy at Call Time

Even if a `PrivacyLayer` was created with one strategy, you can override it
per-call by passing the `strategy` argument to `synthesize_data`.

In [5]:
print("=== Override strategy at call time ===")
synthetic_override = layer.synthesize_data(n_rows=3, strategy="faker")
print(synthetic_override.to_string(index=False))

=== Override strategy at call time ===
          name                      email  age       salary  department  is_active
  Paul Perkins michellehanson@example.net   31 56389.824674 Engineering      False
Michael Thomas  fosterbarbara@example.org   22 65982.969456       Sales       True
  Craig Levine       hahnsean@example.net   25 50857.285911       Sales       True


## 5. Schema Comparison

Synthetic data preserves the original column names and dtypes, ensuring
downstream code works identically on real or synthetic frames.

In [6]:
print("=== Schema comparison ===")
print(f"  Original columns:  {list(df.columns)}")
print(f"  Synthetic columns: {list(synthetic_random.columns)}")
print()
print("  Original dtypes:")
for col, dtype in df.dtypes.items():
    print(f"    {col:15s}: {dtype}")
print("  Synthetic dtypes:")
for col, dtype in synthetic_random.dtypes.items():
    print(f"    {col:15s}: {dtype}")

=== Schema comparison ===
  Original columns:  ['name', 'email', 'age', 'salary', 'department', 'is_active']
  Synthetic columns: ['name', 'email', 'age', 'salary', 'department', 'is_active']

  Original dtypes:
    name           : str
    email          : str
    age            : int64
    salary         : float64
    department     : str
    is_active      : bool
  Synthetic dtypes:
    name           : str
    email          : str
    age            : int64
    salary         : float64
    department     : str
    is_active      : bool


## Analysis

**Goal:** Allow an AI agent to generate synthetic data that preserves the schema of a real dataset but contains no real values, enabling safe prototyping.

**Results:**
- **Random strategy** generates data by sampling from the observed domain of each column. Names and emails are drawn from existing values (not truly new), but numeric columns produce fresh random values within the observed range. The synthetic data has the correct number of rows and matching column structure.
- **Faker strategy** produces contextually realistic values: fake full names (e.g. "Mary Mathis"), plausible email addresses (e.g. "iflores@example.net"), and random numeric values. This is much more realistic for prototyping than the random strategy.
- **Strategy override** works correctly -- even though the layer defaults to "random", passing `strategy="faker"` at call time produces faker-style output.
- **Schema comparison** confirms that column names and dtypes match exactly between original and synthetic DataFrames. This ensures downstream code that works on synthetic data will also work on the real data (modulo values).

**Verdict:** The synthetic data API achieves its goal. An agent can prototype analysis code, build visualizations, or validate pipelines using realistic-looking fake data without any risk of exposing real sensitive records. Both strategies (random and faker) preserve the schema faithfully.